# MSC ESH Final Simulation — Production Pipeline

This notebook is the streamlined production flow for MSC ESH simulation and export.

Execution order is intentionally linear and minimal:
1. Configure inputs and run settings
2. Build geometry and irradiance terms
3. Apply eclipse, structural, and operational masks
4. Compute final ESH and export downstream CSV

Notes:
- `ALBEDO_MODE = "power"` uses cached NASA POWER daily albedo with configurable query cadence.
- If POWER returns missing/fill values for requested dates, fallback `ALBEDO_CONSTANT` is used.


### Setup — Imports and File Paths

Load core libraries and define paths to all STK export files.
All data files live in `data/` relative to this notebook.

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "data"

# Updated to match current STK export filenames
ISS_STATE_PATH  = f"{DATA_DIR}/ISS_Model_J2000_Position_Velocity.csv"
SUN_VEC_PATH    = f"{DATA_DIR}/ISS_Model_Sun_Vector_J2000.csv"
LIGHTING_PATH   = f"{DATA_DIR}/ISS_Model_Lighting_Times.csv"
AZ_EL_MASK_PATH = f"{DATA_DIR}/MSC_Sun_Sensor_Az_El_Mask.csv"
LLA_PATH        = f"{DATA_DIR}/ISS_Model_LLA_Position.csv"   # reserved for future albedo

print("ISS state: ", ISS_STATE_PATH)
print("Sun vector:", SUN_VEC_PATH)
print("Lighting:  ", LIGHTING_PATH)
print("Az-El mask:", AZ_EL_MASK_PATH)


### User Configuration — Face Selection and Exposure Windows

**Edit this section before running the notebook.**

- `PRIMARY_FACE` — which MSC face drives diagnostics and the exported CSV.
- `EXPOSURE_WINDOWS` — list of operational open intervals.  Leave empty (`[]`) to default to always-open (no additional masking beyond eclipse/structure).

Timestamp format: `"%d %b %Y %H:%M:%S.%f"` — identical to the STK export convention.

In [ ]:
# ============================================================
# USER CONFIGURATION — edit before running
# ============================================================

# --- Primary face for diagnostics and CSV export ---
VALID_FACES  = ["+R", "-R", "+T", "-T", "+H", "-H"]
PRIMARY_FACE = "+H"   # change to any of VALID_FACES

if PRIMARY_FACE not in VALID_FACES:
    raise ValueError(
        f"PRIMARY_FACE '{PRIMARY_FACE}' is not a valid face label. "
        f"Choose one of: {VALID_FACES}"
    )

# --- Operational exposure windows ---
# List of (start_str, end_str) pairs using the same timestamp format as STK exports.
# When the list is empty the full simulation period is treated as operationally open.
#
# Example:
#   EXPOSURE_WINDOWS = [
#       ("04 Mar 2026 10:54:00.000", "04 Mar 2026 11:54:00.000"),
#       ("05 Mar 2026 08:00:00.000", "05 Mar 2026 09:30:00.000"),
#   ]
EXPOSURE_WINDOWS = []   # list of ("start", "stop") string pairs, or []

_TS_FMT = "%d %b %Y %H:%M:%S.%f"   # must match STK timestamp convention


# ── Helper: parse, validate, sort, and merge exposure windows ─────────────────
def parse_exposure_windows(exposure_windows, timestamp_format="%d %b %Y %H:%M:%S.%f"):
    """
    Convert EXPOSURE_WINDOWS string pairs to a sorted, merged list of
    (pd.Timestamp, pd.Timestamp) intervals.

    Parameters
    ----------
    exposure_windows : list of (str, str)
        Each element is a (start, stop) pair in timestamp_format.
    timestamp_format : str

    Returns
    -------
    list of (pd.Timestamp, pd.Timestamp)
        Sorted, non-overlapping intervals.  Empty list if input is empty.
    """
    if not exposure_windows:
        return []

    parsed = []
    for i, (s, e) in enumerate(exposure_windows):
        try:
            t_start = pd.to_datetime(s, format=timestamp_format)
        except Exception as exc:
            raise ValueError(
                f"EXPOSURE_WINDOWS[{i}] start '{s}' could not be parsed: {exc}"
            )
        try:
            t_end = pd.to_datetime(e, format=timestamp_format)
        except Exception as exc:
            raise ValueError(
                f"EXPOSURE_WINDOWS[{i}] end '{e}' could not be parsed: {exc}"
            )
        if t_end <= t_start:
            raise ValueError(
                f"EXPOSURE_WINDOWS[{i}]: end must be strictly after start "
                f"(got start={t_start}, end={t_end})"
            )
        parsed.append((t_start, t_end))

    # Sort by start time
    parsed.sort(key=lambda x: x[0])

    # Merge overlapping or touching intervals for robust masking
    merged = [parsed[0]]
    for start, end in parsed[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))

    return merged


# Parse now so format errors surface immediately at config time
_parsed_windows = parse_exposure_windows(EXPOSURE_WINDOWS, _TS_FMT)

print(f"PRIMARY_FACE      : {PRIMARY_FACE}")
print(f"Exposure windows  : {len(_parsed_windows)} interval(s) parsed")


### Albedo Model Configuration

Set `ALBEDO_MODE` to one of three tiers:

| Mode | Description |
|---|---|
| `"power"` | **Tier A (recommended)** — NASA POWER daily surface albedo queried at the ISS sub-satellite lat/lon. Fetches `ALLSKY_SRF_ALB` (or `CLRSKY_SRF_ALB`) via REST API and caches results to `data/cache/`. |
| `"constant"` | **Tier C (fallback)** — fixed `ρ = ALBEDO_CONSTANT` everywhere. |
| `"ceres"` | **Tier B (advanced)** — prepared CERES SYN1deg CSV placed at `data/ceres_albedo_1day.csv`. |

**`POWER_ALBEDO_PARAM`** selects the POWER parameter:
- `"ALLSKY_SRF_ALB"` — all-sky surface albedo (includes cloud contribution).
- `"CLRSKY_SRF_ALB"` — clear-sky surface albedo only.

POWER results are cached in `POWER_CACHE_DIR` as small JSON files so repeated
runs never re-query the same grid cell.  `ALBEDO_CONSTANT` doubles as the
fallback when POWER returns a fill value.


In [ ]:
# --- Albedo model configuration ---

ALBEDO_MODE     = "power"          # "power" | "constant" | "ceres"
ALBEDO_CONSTANT = 0.27             # fixed albedo ("constant" mode) + POWER fallback

# NASA POWER (Tier A) settings
POWER_ALBEDO_PARAM       = "ALLSKY_SRF_ALB"   # or "CLRSKY_SRF_ALB"
POWER_CACHE_DIR          = f"{DATA_DIR}/cache"
POWER_QUERY_INTERVAL_MIN = 1440                # 360=4 queries/day, 1440=1 query/day

# CERES CSV path (Tier B — optional)
CERES_ALBEDO_PATH = f"{DATA_DIR}/ceres_albedo_1day.csv"

# Small floor to avoid divide-by-zero edge behavior
EPS = 1e-12

print("ALBEDO_MODE              :", ALBEDO_MODE)
print("ALBEDO_CONSTANT          :", ALBEDO_CONSTANT)
if ALBEDO_MODE == "power":
    print("POWER_ALBEDO_PARAM       :", POWER_ALBEDO_PARAM)
    print("POWER_CACHE_DIR          :", POWER_CACHE_DIR)
    print("POWER_QUERY_INTERVAL_MIN :", POWER_QUERY_INTERVAL_MIN)


### Effective Albedo Source

Prepare `rho_eff_series` based on the configured mode.

- **`"power"`** — load the LLA file, snap sub-satellite lat/lon to the nearest
  0.5° POWER grid cell, fetch (or retrieve from cache) one daily albedo value
  per unique cell, and return a per-timestep Series.
- **`"constant"`** — nothing is loaded here; the value is attached to `df` later.
- **`"ceres"`** — the prepared CSV is loaded here and joined by time downstream.


In [ ]:
# --- Effective albedo source rho_eff(t) ---

if ALBEDO_MODE == "constant":
    rho_eff_series = None   # placeholder; attached to df after df is built

elif ALBEDO_MODE == "power":
    # Load LLA to get sub-satellite track timestamps and coordinates.
    # This does NOT depend on df — it only needs the LLA CSV.
    _lla = pd.read_csv(LLA_PATH)
    _lla["t"] = pd.to_datetime(_lla["Time (UTCG)"], format="%d %b %Y %H:%M:%S.%f")

    from power_albedo import build_power_rho_series
    rho_eff_series = build_power_rho_series(
        _lla,
        cache_dir=POWER_CACHE_DIR,
        parameter=POWER_ALBEDO_PARAM,
        fallback=ALBEDO_CONSTANT,
        query_interval_minutes=POWER_QUERY_INTERVAL_MIN,
    )
    print("rho_eff summary (POWER):")
    print(rho_eff_series["rho_eff"].describe())

elif ALBEDO_MODE == "ceres":
    ceres_df = pd.read_csv(CERES_ALBEDO_PATH)
    ceres_df["t"] = pd.to_datetime(ceres_df["t"])
    ceres_df = ceres_df.sort_values("t").reset_index(drop=True)
    if "rho_eff" not in ceres_df.columns:
        raise KeyError(f"Expected 'rho_eff' in {CERES_ALBEDO_PATH}")
    rho_eff_series = ceres_df[["t", "rho_eff"]].copy()

else:
    raise ValueError(f"Unsupported ALBEDO_MODE: {ALBEDO_MODE!r}")

print("Prepared albedo source for mode:", ALBEDO_MODE)


### Load STK Exports

Read the ISS state (`ISS_Model_J2000_Position_Velocity.csv`) and Sun vector
(`ISS_Model_Sun_Vector_J2000.csv`) CSVs exported from STK. Print shapes to confirm both loaded.

In [ ]:
iss_df = pd.read_csv(ISS_STATE_PATH)
sun_df = pd.read_csv(SUN_VEC_PATH)

print("ISS columns:")
print(iss_df.columns)

print("\nSun columns:")
print(sun_df.columns)

print("\nISS shape:", iss_df.shape)
print("Sun shape:", sun_df.shape)

### Rename Columns and Merge

Normalise STK column names to short internal names and merge ISS state + Sun vector
into a single dataframe keyed by time.

In [ ]:
# Rename ISS columns
iss_df = iss_df.rename(columns={
    'Time (UTCG)': 'time',
    'x (km)': 'r_x',
    'y (km)': 'r_y',
    'z (km)': 'r_z',
    'vx (km/sec)': 'v_x',
    'vy (km/sec)': 'v_y',
    'vz (km/sec)': 'v_z'
})

# Rename Sun columns
sun_df = sun_df.rename(columns={
    'Time (UTCG)': 'time',
    'x (km)': 'sun_x',
    'y (km)': 'sun_y',
    'z (km)': 'sun_z'
})

# Merge
df = pd.merge(iss_df, sun_df, on='time')

print(df.columns)
print(df.shape)

df.head()

### Parse Timestamps and Compute Timestep

Convert STK time strings (`"4 Mar 2026 18:00:00.000"`) to pandas datetime,
sort, and compute `dt` (seconds) between rows for later time-integration.

In [ ]:
df["t"] = pd.to_datetime(df["time"], format="%d %b %Y %H:%M:%S.%f")

df = df.sort_values("t").reset_index(drop=True)

df["dt"] = df["t"].diff().dt.total_seconds()
df.loc[0, "dt"] = df.loc[1, "dt"]

df[["time", "t", "dt"]].head()

### Build Operational Exposure Mask

Create `df["exposure_open_factor"]` from `EXPOSURE_WINDOWS`.

- **1.0** when the timestep falls inside any configured exposure-open interval.
- **0.0** otherwise.

If `EXPOSURE_WINDOWS` is empty the column is set to **1.0** everywhere (always open — no additional operational masking beyond eclipse/structure).

This factor is applied on top of `eclipse_factor` and `structural_factor` in the masked irradiance computation.

In [ ]:
# --- Build operational exposure mask ---
# Requires df["t"] to exist (run timestamp parsing cell above first).

def build_exposure_mask(df_t_col, parsed_windows):
    """
    Return a float array (1.0 / 0.0) indicating whether each timestep
    falls inside any operationally open exposure interval.

    Parameters
    ----------
    df_t_col : pd.Series of pd.Timestamp
    parsed_windows : list of (pd.Timestamp, pd.Timestamp)
        Output of parse_exposure_windows().

    Returns
    -------
    np.ndarray of float64, same length as df_t_col
    """
    if not parsed_windows:
        # No windows configured — always open
        return np.ones(len(df_t_col), dtype=float)

    t_arr = df_t_col.to_numpy()   # numpy datetime64
    mask  = np.zeros(len(df_t_col), dtype=bool)

    for t_start, t_end in parsed_windows:
        s64 = np.datetime64(t_start)
        e64 = np.datetime64(t_end)
        mask |= (t_arr >= s64) & (t_arr <= e64)

    return mask.astype(float)


df["exposure_open_factor"] = build_exposure_mask(df["t"], _parsed_windows)

# ── Exposure window summary / sanity check ─────────────────────────────────────
sim_start    = df["t"].iloc[0]
sim_end      = df["t"].iloc[-1]
sim_dur_h    = (sim_end - sim_start).total_seconds() / 3600.0
open_frac    = df["exposure_open_factor"].mean()
open_hours   = open_frac * sim_dur_h
n_open_steps = int(df["exposure_open_factor"].sum())

print("=" * 54)
print("CONFIGURATION SUMMARY")
print("=" * 54)
print(f"  Primary face       : {PRIMARY_FACE}")
print(f"  Sim start          : {sim_start}")
print(f"  Sim end            : {sim_end}")
print(f"  Sim duration       : {sim_dur_h:.2f} h")
print()
if _parsed_windows:
    print(f"  Exposure windows   : {len(_parsed_windows)} interval(s)")
    for i, (s, e) in enumerate(_parsed_windows):
        dur_h = (e - s).total_seconds() / 3600.0
        print(f"    [{i}]  {s}  ->  {e}  ({dur_h:.2f} h)")
else:
    print("  Exposure windows   : none — full sim treated as operationally open")
print()
print(f"  Open timesteps     : {n_open_steps} / {len(df)}")
print(f"  Open fraction      : {open_frac * 100:.1f}%")
print(f"  Open duration      : {open_hours:.2f} h")
print("=" * 54)


### Attach Albedo to Master Dataframe

In `"constant"` mode: fill `rho_eff` column with the configured constant.
In `"ceres"` mode: join the loaded CERES series to `df` by nearest timestamp.

In [ ]:
# --- Attach rho_eff to df ---

if ALBEDO_MODE == "constant":
    df["rho_eff"] = ALBEDO_CONSTANT

elif ALBEDO_MODE in ("power", "ceres"):
    # rho_eff_series is a DataFrame with ['t', 'rho_eff'] built in the cell above.
    # Merge to df by nearest timestamp so every simulation row gets an albedo value.
    df = pd.merge_asof(
        df.sort_values("t"),
        rho_eff_series.sort_values("t"),
        on="t",
        direction="nearest",
    )
    df["rho_eff"] = df["rho_eff"].fillna(ALBEDO_CONSTANT)

print(f"rho_eff summary  [{ALBEDO_MODE}]:")
print(df["rho_eff"].describe())
df[["t", "rho_eff"]].head()


### Build LVLH Frame

Compute the three LVLH unit axes from ISS inertial position and velocity:
- **R̂**: radial (Earth → ISS)
- **Ĥ**: orbit normal (angular momentum direction)
- **T̂**: along-track (Ĥ × R̂)

In [ ]:
r = df[["r_x","r_y","r_z"]].to_numpy()
v = df[["v_x","v_y","v_z"]].to_numpy()

def unit(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / n

Rhat = unit(r)

H = np.cross(r, v)
Hhat = unit(H)

That = unit(np.cross(Hhat, Rhat))

print(np.mean(np.linalg.norm(Rhat, axis=1)))
print(np.mean(np.linalg.norm(Hhat, axis=1)))
print(np.mean(np.linalg.norm(That, axis=1)))

### Sun Direction Vector (J2000)

Compute the unit vector from ISS toward the Sun by subtracting ISS position
from the STK Sun position and normalising.

In [ ]:
sun_pos = df[["sun_x","sun_y","sun_z"]].to_numpy()
iss_pos = r

s_vec = sun_pos - iss_pos
s_hat = unit(s_vec)

print(np.mean(np.linalg.norm(s_hat, axis=1)))

### Project Sun Direction into LVLH

Project the inertial Sun unit vector onto the LVLH basis to get
`(s_R, s_T, s_H)` — the Sun direction expressed in the body frame.

In [ ]:
s_R = np.sum(s_hat * Rhat, axis=1)
s_T = np.sum(s_hat * That, axis=1)
s_H = np.sum(s_hat * Hhat, axis=1)

s_lvlh = np.vstack([s_R, s_T, s_H]).T

print(np.mean(np.linalg.norm(s_lvlh, axis=1)))

df["s_R"] = s_R
df["s_T"] = s_T
df["s_H"] = s_H

df[["time","s_R","s_T","s_H"]].head()

### Earth Direction and Albedo Geometry

Compute the Earth view factor `F_earth` (solid angle of the Earth disk as seen from the ISS)
and a `day_factor` approximating whether the Earth below is sunlit.
These scale the albedo irradiance term for each face.

In [ ]:
# --- Earth direction + albedo geometry terms ---

R_EARTH_KM = 6378.137

# In this notebook, +R points away from Earth, so the unit vector toward Earth is -Rhat.
# In LVLH component form, that is simply [-1, 0, 0] at every timestep.
e_hat_lvlh = np.column_stack([
    -np.ones(len(df)),
    np.zeros(len(df)),
    np.zeros(len(df))
])

# Spacecraft geocentric distance and altitude
r_mag = np.linalg.norm(r, axis=1)          # km
alt_km = r_mag - R_EARTH_KM

# Earth apparent half-angle as seen from the spacecraft
earth_half_angle = np.arcsin(np.clip(R_EARTH_KM / r_mag, 0.0, 1.0))

# First-order Earth view factor
# Earth fills a larger fraction of the sky at lower altitude
F_earth = np.sin(earth_half_angle) ** 2

# Simple day/night factor for the Earth below the spacecraft
# If the Sun is on the Earth side of the local frame, the visible Earth contributes reflected light
day_factor = np.clip(-s_R, 0.0, 1.0)

df["alt_km"] = alt_km
df["F_earth"] = F_earth
df["day_factor"] = day_factor

df[["t", "alt_km", "F_earth", "day_factor"]].head()

### MSC Face Normals  *(Placeholder — generic LVLH box)*

Define six unit face normals in the LVLH frame.
**These are a stand-in box model and do not yet reflect the actual MSC panel geometry.**
Replace with real MSC face orientations (tilt angles, cant, mounting face) when available.

| Face | LVLH direction |
|---|---|
| +R | Radially outward (away from Earth) |
| -R | Nadir (toward Earth) |
| +T | Along-track |
| -T | Anti-velocity |
| +H | Orbit normal |
| -H | Anti-orbit normal |


In [ ]:
face_normals = {
    "+R": np.array([1.0, 0.0, 0.0]),
    "-R": np.array([-1.0, 0.0, 0.0]),
    "+T": np.array([0.0, 1.0, 0.0]),
    "-T": np.array([0.0,-1.0, 0.0]),
    "+H": np.array([0.0, 0.0, 1.0]),
    "-H": np.array([0.0, 0.0,-1.0]),
}

### Unmasked Irradiance per Face

Compute `I_direct` and `I_albedo` for each face.
Eclipse and structural masking are applied in later cells — these are the raw geometric terms.

In [ ]:
# --- Compute direct + albedo + total irradiance for each face ---

AM0 = 1361.0  # W/m^2

for face, n in face_normals.items():
    n = np.asarray(n, dtype=float)

    # Direct solar term
    cos_sun = np.clip(
        n[0] * s_R +
        n[1] * s_T +
        n[2] * s_H,
        0.0,
        1.0
    )
    I_direct = AM0 * cos_sun

    # Earth albedo term
    # e_hat_lvlh = direction toward Earth in LVLH coordinates
    cos_earth = np.clip(
        n[0] * e_hat_lvlh[:, 0] +
        n[1] * e_hat_lvlh[:, 1] +
        n[2] * e_hat_lvlh[:, 2],
        0.0,
        1.0
    )
    I_albedo = df["rho_eff"].to_numpy() * AM0 * df["F_earth"].to_numpy() * df["day_factor"].to_numpy() * cos_earth

    # Total
    I_total = I_direct + I_albedo

    df[f"cos_sun_{face}"] = cos_sun
    df[f"cos_earth_{face}"] = cos_earth
    df[f"I_direct_{face}"] = I_direct
    df[f"I_albedo_{face}"] = I_albedo
    df[f"I_{face}"] = I_total   # keeps downstream notebook compatibility

df[[c for c in df.columns if c.startswith("I_")][:12]].head()

### Parse STK Lighting Report

Parse `ISS_Model_Lighting_Times.csv`, which contains three tables with the same header
(separated by "Global Statistics" blocks):

| Table index | Content | Typical duration |
|---|---|---|
| 0 | Sunlight intervals | ~60 min |
| 1 | Penumbra transitions | ~12 sec |
| 2 | Umbra / shadow intervals | ~32 min |


In [ ]:
from pathlib import Path
from io import StringIO

# Parse the STK lighting report (ISS_Model_Lighting_Times.csv).
# The file contains THREE tables with the same header, separated by
# "Global Statistics" blocks, in order:
#   Table 0 — Sunlight intervals
#   Table 1 — Penumbra intervals  (~12 sec each; transitions)
#   Table 2 — Umbra/Shadow intervals  (~32 min each)

STK_TIME_FMT = "%d %b %Y %H:%M:%S.%f"
HEADER       = '"Start Time (UTCG)","Stop Time (UTCG)","Duration (sec)"'

raw_lines    = Path(LIGHTING_PATH).read_text().splitlines()
header_idxs  = [i for i, l in enumerate(raw_lines) if l.strip() == HEADER]
print(f"Found {len(header_idxs)} table(s) in {LIGHTING_PATH}")

def extract_lighting_table(table_index):
    """Return a DataFrame for the Nth lighting table (0=Sunlight, 1=Penumbra, 2=Umbra)."""
    start_i = header_idxs[table_index]
    end_i   = start_i + 1
    while end_i < len(raw_lines):
        line = raw_lines[end_i].strip()
        if line == "" or line.startswith("Global Statistics"):
            break
        end_i += 1
    return pd.read_csv(StringIO("\n".join(raw_lines[start_i:end_i])))

def parse_interval_df(df_raw):
    """Add parsed 'start' and 'stop' datetime columns."""
    df_out = df_raw.copy()
    df_out["start"] = pd.to_datetime(df_out["Start Time (UTCG)"], format=STK_TIME_FMT)
    df_out["stop"]  = pd.to_datetime(df_out["Stop Time (UTCG)"],  format=STK_TIME_FMT)
    return df_out[["start", "stop"]]

sunlight_df = parse_interval_df(extract_lighting_table(0))
penumbra_df = parse_interval_df(extract_lighting_table(1))
umbra_df    = parse_interval_df(extract_lighting_table(2))

print(f"Sunlight  intervals: {len(sunlight_df)}")
print(f"Penumbra  intervals: {len(penumbra_df)}")
print(f"Umbra     intervals: {len(umbra_df)}")
umbra_df.head()


### Eclipse Factor per Timestep

Assign a per-timestep `eclipse_factor` based on lighting state:

| State | eclipse_factor |
|---|---|
| Sunlight | 1.0 |
| Penumbra | 0.5 *(placeholder — transitions are ~12 sec, low ESH impact)* |
| Umbra | 0.0 |

`sun_factor` is kept as an alias for downstream CSV compatibility.


In [ ]:
# Build per-timestep lighting state and eclipse factor.
#
# Lighting priority: Umbra > Penumbra > Sunlight
#
# eclipse_factor:
#   Sunlight  -> 1.0
#   Penumbra  -> 0.5  (fractional placeholder; refine if penumbra model improves)
#   Umbra     -> 0.0

t = df["t"].to_numpy()

in_umbra    = np.zeros(len(df), dtype=bool)
in_penumbra = np.zeros(len(df), dtype=bool)

for s, e in zip(umbra_df["start"].to_numpy(), umbra_df["stop"].to_numpy()):
    in_umbra |= (t >= s) & (t <= e)

for s, e in zip(penumbra_df["start"].to_numpy(), penumbra_df["stop"].to_numpy()):
    in_penumbra |= (t >= s) & (t <= e)

# Named state
lighting_state = np.where(in_umbra, "Umbra",
                 np.where(in_penumbra, "Penumbra", "Sunlight"))

# Eclipse factor
eclipse_factor = np.ones(len(df))
eclipse_factor[in_penumbra] = 0.5
eclipse_factor[in_umbra]    = 0.0

# Keep sun_factor as alias so downstream CSV stays compatible
df["lighting_state"] = lighting_state
df["eclipse_factor"] = eclipse_factor
df["sun_factor"]     = eclipse_factor   # alias — do not remove

print("Umbra    timesteps:", int(in_umbra.sum()))
print("Penumbra timesteps:", int(in_penumbra.sum()))
print("Sunlight timesteps:", int((~in_umbra & ~in_penumbra).sum()))
df[["t", "lighting_state", "eclipse_factor"]].head(20)


### Az-El Mask — Parse Structural Blocking Zones

Parse `MSC_Sun_Sensor_Az_El_Mask.csv` (STK export) into:
- **Exclusion Zones**: sky regions blocked by ISS structure.
- **Inclusion Zones**: transparent gaps inside a specific exclusion zone (e.g. holes in a solar array bracket).

The file contains **6 exclusion zones and 57 inclusion zones**:

| Zone | Az range | El range | Inclusions | Description |
|---|---|---|---|---|
| EZ1 | [-135, +135] | [+34.3, +34.4] | 0 | Thin seam band at upper El boundary |
| EZ2 | [-135, +135] | [-34.4, -34.3] | 0 | Thin seam band at lower El boundary |
| EZ3 | [-46, +46] | [-35.8, +35.8] | 15 | Forward/aft ISS body region |
| EZ4 | [-134, +134] | [-35.8, +35.8] | 0 | Main ISS truss rectangle |
| EZ5 | [+44, +136] | [-35.8, +35.8] | 21 | Right-side solar array quadrant |
| EZ6 | [-136, -44] | [-35.8, +35.8] | 21 | Left-side solar array quadrant |

**Note on beta angle:** All exclusion zones are below El = 35.84°. On high-beta-angle days
(beta > ~36°) the Sun never enters any exclusion zone and blocking = 0% — which is physically
correct. The 4–5 Mar 2026 simulation has beta ≈ 42°. Run a low-beta date to exercise blocking.


In [ ]:
import re

def parse_az_el_mask(mask_path):
    """
    Parse the STK Az-El mask CSV.

    Returns
    -------
    exclusion_zones : list of np.ndarray, shape (N, 2), columns [Az_deg, El_deg]
    inclusion_map   : dict {excl_idx (0-based): [polygon_array, ...]}
    """
    with open(mask_path, "r") as f:
        lines = f.readlines()

    exclusion_zones = []
    inclusion_map   = {}

    current_type  = None   # "exclusion" or "inclusion"
    current_excl  = None   # parent exclusion zone index (0-based)
    current_pts   = []

    excl_re = re.compile(r'"Exclusion Zone (\d+)', re.IGNORECASE)
    incl_re = re.compile(r'"Inclusion Zone \d+ \(Inside Exclusion Zone (\d+)\)',
                          re.IGNORECASE)
    data_re = re.compile(r'\d+,(-?\d+\.?\d*),(-?\d+\.?\d*)')

    def flush():
        if not current_pts:
            return
        pts = np.array(current_pts, dtype=float)
        if current_type == "exclusion":
            exclusion_zones.append(pts)
        elif current_type == "inclusion":
            inclusion_map.setdefault(current_excl, []).append(pts)

    for line in lines:
        line = line.strip()
        m_excl = excl_re.search(line)
        m_incl = incl_re.search(line)
        m_data = data_re.match(line)

        if m_excl:
            flush()
            current_type  = "exclusion"
            current_excl  = int(m_excl.group(1)) - 1   # convert to 0-based
            current_pts   = []
        elif m_incl:
            flush()
            current_type  = "inclusion"
            current_excl  = int(m_incl.group(1)) - 1   # 0-based parent index
            current_pts   = []
        elif m_data:
            current_pts.append([float(m_data.group(1)), float(m_data.group(2))])

    flush()   # save the last zone

    return exclusion_zones, inclusion_map

exclusion_zones, inclusion_map = parse_az_el_mask(AZ_EL_MASK_PATH)

print(f"Parsed {len(exclusion_zones)} exclusion zone(s):")
for i, z in enumerate(exclusion_zones):
    n_inc = len(inclusion_map.get(i, []))
    az_r  = f"Az [{z[:,0].min():.1f}, {z[:,0].max():.1f}]"
    el_r  = f"El [{z[:,1].min():.1f}, {z[:,1].max():.1f}]"
    print(f"  EZ{i+1}: {len(z)} vertices | {az_r} | {el_r} | {n_inc} inclusion zone(s)")


### Sun Az/El in Sensor Frame

Convert LVLH Sun direction components `(s_R, s_T, s_H)` to azimuth/elevation
for comparison against the STK Az-El mask.

**Convention (adjust `AZ_OFFSET_DEG` if sensor mount orientation differs):**
| Symbol | Definition |
|--------|-----------|
| El | arcsin(s_H) — 0° in orbital plane, +90° at +H boresight |
| Az | atan2(s_T, s_R) — 0° toward +R (radially out), positive toward +T |

This is consistent with the LVLH frame already computed in this notebook.


In [ ]:
# Compute Sun azimuth and elevation in the sensor/LVLH frame.
#
# Adjust AZ_OFFSET_DEG if the STK sensor is rotated relative to the LVLH +R direction.
AZ_OFFSET_DEG = 0.0

sun_el_rad = np.arcsin(np.clip(df["s_H"].to_numpy(), -1.0, 1.0))
sun_az_rad = np.arctan2(df["s_T"].to_numpy(), df["s_R"].to_numpy())

sun_el_deg = np.degrees(sun_el_rad)
sun_az_deg = np.degrees(sun_az_rad) + AZ_OFFSET_DEG

# Wrap Az to [-180, 180)
sun_az_deg = (sun_az_deg + 180.0) % 360.0 - 180.0

df["sun_az_deg"] = sun_az_deg
df["sun_el_deg"] = sun_el_deg

print(f"Sun Az: {sun_az_deg.min():.1f} to {sun_az_deg.max():.1f} deg")
print(f"Sun El: {sun_el_deg.min():.1f} to {sun_el_deg.max():.1f} deg")
df[["t", "sun_az_deg", "sun_el_deg"]].head()


### Structural Blocking Factor (Az-El Mask)

For each timestep:
1. Convert the LVLH Sun direction to `(Az, El)` in the sensor frame.
2. Test whether that direction falls inside an **Exclusion Zone** (ISS structure blocking).
3. If it also falls inside a nested **Inclusion Zone** (a gap in the structure), treat it as unblocked.

```
blocked = in_any_exclusion  AND NOT in_any_inclusion
structural_factor = 0 if blocked, else 1
```

Only **direct solar** irradiance is affected. Albedo is assumed diffuse and not blocked.


In [ ]:
from matplotlib.path import Path as MplPath

def make_path(polygon_pts):
    """Closed matplotlib Path from Nx2 array of [Az, El] points."""
    return MplPath(polygon_pts, closed=True)

# Pre-build all exclusion + inclusion paths
excl_paths     = [make_path(z) for z in exclusion_zones]
incl_paths_map = {
    i: [make_path(z) for z in zones]
    for i, zones in inclusion_map.items()
}

# Batch point-in-polygon for all timesteps
sun_pts = np.column_stack([df["sun_az_deg"].to_numpy(),
                           df["sun_el_deg"].to_numpy()])

in_any_exclusion = np.zeros(len(df), dtype=bool)
in_any_inclusion = np.zeros(len(df), dtype=bool)

for i, ep in enumerate(excl_paths):
    in_excl_i = ep.contains_points(sun_pts)
    in_any_exclusion |= in_excl_i

    # Check inclusion zones linked to this exclusion zone
    for ip in incl_paths_map.get(i, []):
        in_any_inclusion |= (in_excl_i & ip.contains_points(sun_pts))

# Final blocking decision
structural_blocked = in_any_exclusion & (~in_any_inclusion)
structural_factor  = np.where(structural_blocked, 0.0, 1.0)

df["structural_blocked"] = structural_blocked
df["structural_factor"]  = structural_factor

print("Structural blocking stats:")
print(f"  In exclusion zone:  {int(in_any_exclusion.sum())} / {len(df)} timesteps")
print(f"  In inclusion zone:  {int(in_any_inclusion.sum())} / {len(df)} timesteps")
print(f"  Net blocked:        {int(structural_blocked.sum())} / {len(df)} timesteps"
      f"  ({structural_blocked.mean()*100:.1f}%)")
df[["t", "sun_az_deg", "sun_el_deg", "structural_blocked", "structural_factor"]].head(15)


### Final Irradiance with Eclipse, Structural, and Operational Masking

Apply all three masking layers to produce the final per-face irradiance and ESH:

```
I_direct_masked = I_direct × eclipse_factor × structural_factor × exposure_open_factor
I_albedo_masked = I_albedo × eclipse_factor × exposure_open_factor
                  (diffuse reflected light — not blocked by structure,
                   but gated by the same operational exposure window)
I_total_masked  = I_direct_masked + I_albedo_masked
ESH             = Σ(I_total_masked × dt) / AM0 / 3600
```

`df["I_{face}"]` is overwritten here so all downstream references use the fully masked value.  
`exposure_open_factor = 1.0` everywhere when `EXPOSURE_WINDOWS = []` (no change from old behaviour).

In [ ]:
# --- Recompute ESH with eclipse masking, structural masking, AND exposure masking ---
#
# Masking philosophy:
#   Direct solar  : eclipse_factor x structural_factor x exposure_open_factor
#   Albedo        : eclipse_factor x exposure_open_factor
#                   (diffuse Earth-reflected light is not blocked by ISS structure,
#                    but is gated by the same operational exposure window)
#
# exposure_open_factor = 1.0 when the exposure window is open, 0.0 otherwise.
# If EXPOSURE_WINDOWS is empty, exposure_open_factor is 1.0 everywhere (no change).

esh_results_eclipse = {}

_ef  = df["eclipse_factor"].to_numpy()
_sf  = df["structural_factor"].to_numpy()
_xf  = df["exposure_open_factor"].to_numpy()   # operational exposure mask

for name in face_normals.keys():

    I_direct_masked = (
        df[f"I_direct_{name}"].to_numpy()
        * _ef
        * _sf
        * _xf
    )
    I_albedo_masked = (
        df[f"I_albedo_{name}"].to_numpy()
        * _ef
        * _xf
    )
    I_total_masked = I_direct_masked + I_albedo_masked

    energy = np.sum(I_total_masked * df["dt"].to_numpy())
    esh    = energy / AM0 / 3600.0

    esh_results_eclipse[name] = esh

    # Store columns for downstream use and diagnostics
    df[f"I_direct_masked_{name}"] = I_direct_masked
    df[f"I_albedo_masked_{name}"] = I_albedo_masked
    df[f"I_total_masked_{name}"]  = I_total_masked
    df[f"I_{name}"]               = I_total_masked   # keep downstream compatibility

esh_results_eclipse

# ── Per-face summary for PRIMARY_FACE ────────────────────────────────────────
_pf = PRIMARY_FACE
_I_pf   = df[f"I_total_masked_{_pf}"].to_numpy()
_dt_pf  = df["dt"].to_numpy()
_esh_pf = esh_results_eclipse[_pf]

print()
print(f"PRIMARY_FACE ({_pf}) summary after all masking:")
print(f"  Max masked irradiance  : {_I_pf.max():.1f} W/m²")
print(f"  Mean masked irradiance : {_I_pf.mean():.1f} W/m²")
print(f"  Final cumulative ESH   : {_esh_pf:.4f} h")


### Cumulative ESH

Compute running integral of masked total irradiance for each face.
Stored as `ESH_{face}_cum` for plotting.

In [ ]:
# --- Cumulative ESH using masked total irradiance ---

for name in face_normals.keys():
    I  = df[f"I_total_masked_{name}"].to_numpy()
    dt = df["dt"].to_numpy()
    cum_energy = np.cumsum(I * dt)
    df[f"ESH_{name}_cum"] = cum_energy / AM0 / 3600.0


### Final ESH Summary

Tabulate the 1-day ESH for each face, sorted descending.  
These are fully masked values: eclipse × structural × operational exposure.

In [ ]:
esh_summary = (
    pd.Series(esh_results_eclipse, name="ESH (hours)")
    .sort_values(ascending=False)
    .to_frame()
)

esh_summary

### Visualizations

#### Az/El Frame Diagnostic

Sun Az/El trajectory overlaid on structural exclusion and inclusion zone polygons.

- If the sun track never enters a red exclusion zone, 0 % blocking is correct for this period.
- If the trajectory looks wrong, adjust `AZ_OFFSET_DEG` in the Sun Az/El cell or check the boresight assumption.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(13, 7))

# --- Exclusion zones ---
ez_colors = ["#e74c3c", "#e67e22", "#8e44ad", "#2980b9", "#27ae60", "#f39c12"]
for i, pts in enumerate(exclusion_zones):
    poly = MplPolygon(pts, closed=True,
                      facecolor=ez_colors[i % len(ez_colors)], alpha=0.25,
                      edgecolor=ez_colors[i % len(ez_colors)], linewidth=1.5,
                      label=f"Exclusion Zone {i+1}")
    ax.add_patch(poly)

# --- Inclusion zones (gaps in structure) ---
for excl_idx, inc_list in inclusion_map.items():
    for j, pts in enumerate(inc_list):
        poly = MplPolygon(pts, closed=True,
                          facecolor="white", alpha=0.85,
                          edgecolor="green", linewidth=1.0, linestyle="--",
                          label="Inclusion zone" if (excl_idx == 0 and j == 0) else "_")
        ax.add_patch(poly)

# --- Sun trajectory ---
sunlit = df["sun_factor"] > 0
sc = ax.scatter(df.loc[sunlit,  "sun_az_deg"], df.loc[sunlit,  "sun_el_deg"],
                c=df.loc[sunlit, "t"].astype("int64"),
                cmap="plasma", s=4, zorder=5, label="Sun (sunlit)")
ax.scatter(df.loc[~sunlit, "sun_az_deg"], df.loc[~sunlit, "sun_el_deg"],
           color="steelblue", s=3, alpha=0.4, zorder=4, label="Sun (eclipse)")
ax.plot(df["sun_az_deg"].iloc[0], df["sun_el_deg"].iloc[0],
        "k^", markersize=8, zorder=6, label="Start")

plt.colorbar(sc, ax=ax, label="Time (earlier → later)")
ax.set_xlabel("Sun Azimuth in Sensor Frame (deg)")
ax.set_ylabel("Sun Elevation in Sensor Frame (deg)")
ax.set_title("Sun Az/El Trajectory vs Az-El Mask Exclusion Zones")
ax.set_xlim(-185, 185)
ax.set_ylim(-95, 95)
ax.axhline(0, color="gray", lw=0.5, linestyle=":")
ax.axvline(0, color="gray", lw=0.5, linestyle=":")
ax.grid(True, alpha=0.3)
handles, labels = ax.get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels):
    if l not in seen:
        seen[l] = h
ax.legend(seen.values(), seen.keys(), loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("outputs/fig_azel_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Sun Az range: {df['sun_az_deg'].min():.1f}° to {df['sun_az_deg'].max():.1f}°")
print(f"Sun El range: {df['sun_el_deg'].min():.1f}° to {df['sun_el_deg'].max():.1f}°")
for i, z in enumerate(exclusion_zones):
    print(f"  EZ{i+1}: Az [{z[:,0].min():.1f}, {z[:,0].max():.1f}]  "
          f"El [{z[:,1].min():.1f}, {z[:,1].max():.1f}]")


#### Irradiance Diagnostics

Six-panel sanity check confirming all masking layers are applied correctly:
1. Sun elevation + eclipse factor
2. Structural blocking flag
3. Operational exposure window
4. Direct irradiance before vs after all masking
5. Total irradiance (direct + albedo) after all masking
6. Cumulative ESH by face

In [ ]:
DIAG_FACE = PRIMARY_FACE

_I_direct_masked = (
    df[f"I_direct_{DIAG_FACE}"].to_numpy()
    * df["eclipse_factor"].to_numpy()
    * df["structural_factor"].to_numpy()
    * df["exposure_open_factor"].to_numpy()
)

def _fmt_ax(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    import matplotlib
    plt.setp(ax.get_xticklabels(), rotation=30)

# 1: Sun elevation + eclipse factor
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df["t"], df["sun_el_deg"], color="gold", label="Sun El (deg)")
ax2 = ax.twinx()
ax2.plot(df["t"], df["eclipse_factor"], color="steelblue", alpha=0.6, label="eclipse_factor")
ax.set_ylabel("Sun Elevation (deg)"); ax2.set_ylabel("Eclipse Factor")
ax.set_title("Sun Elevation & Eclipse Factor")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")
_fmt_ax(ax); plt.tight_layout()
plt.savefig("outputs/fig_sun_eclipse.png", dpi=150, bbox_inches="tight")
plt.show()

# 2: Structural blocking flag
fig, ax = plt.subplots(figsize=(13, 3))
ax.fill_between(df["t"], df["structural_blocked"].astype(float),
                color="tomato", alpha=0.6, label="blocked by structure")
ax.set_ylabel("Blocked (1=yes)"); ax.set_ylim(-0.05, 1.25)
ax.set_title("Structural Blocking Flag (Az-El Mask)")
ax.legend(); _fmt_ax(ax); plt.tight_layout()
plt.savefig("outputs/fig_structural_blocking.png", dpi=150, bbox_inches="tight")
plt.show()

# 3: Exposure window mask
_xf = df["exposure_open_factor"].to_numpy()
_win_label = f"{len(EXPOSURE_WINDOWS)} interval(s)" if EXPOSURE_WINDOWS else "always open"
fig, ax = plt.subplots(figsize=(13, 3))
ax.fill_between(df["t"], _xf, where=(_xf > 0.5), color="mediumseagreen", alpha=0.55, label="open")
ax.fill_between(df["t"], 1.0, where=(_xf < 0.5), color="lightcoral",    alpha=0.35, label="closed")
ax.plot(df["t"], _xf, color="darkgreen", lw=0.8)
ax.set_ylabel("Exposure Open (1=open)"); ax.set_ylim(-0.1, 1.3)
ax.set_title(f"Operational Exposure Window Mask  ({_win_label})")
ax.legend(); _fmt_ax(ax); plt.tight_layout()
plt.savefig("outputs/fig_exposure_mask.png", dpi=150, bbox_inches="tight")
plt.show()

# 4: Direct irradiance before vs after masking
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df["t"], df[f"I_direct_{DIAG_FACE}"], color="orange", lw=0.8, label="direct (no mask)")
ax.plot(df["t"], _I_direct_masked, color="navy", lw=1.0, label="direct (all masks)")
ax.set_ylabel("W/m²")
ax.set_title(f"Face {DIAG_FACE}: Direct Irradiance Before vs After All Masking")
ax.legend(); _fmt_ax(ax); plt.tight_layout()
plt.savefig("outputs/fig_direct_irradiance.png", dpi=150, bbox_inches="tight")
plt.show()

# 5: Total irradiance after masking
_I_total_masked = df[f"I_{DIAG_FACE}"].to_numpy()
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df["t"], _I_direct_masked, color="navy",    lw=0.8, label="direct (all masks)")
ax.plot(df["t"], _I_total_masked,  color="crimson", lw=1.0, label="total = direct + albedo (all masks)")
ax.set_ylabel("W/m²")
ax.set_title(f"Face {DIAG_FACE}: Total Irradiance After All Masking")
ax.legend(); _fmt_ax(ax); plt.tight_layout()
plt.savefig("outputs/fig_total_irradiance.png", dpi=150, bbox_inches="tight")
plt.show()

# 6: Cumulative ESH by face
fig, ax = plt.subplots(figsize=(13, 4))
for name in face_normals:
    col = f"ESH_{name}_cum"
    if col in df.columns:
        lw = 2.2 if name == DIAG_FACE else 1.0
        ax.plot(df["t"], df[col], label=name, linewidth=lw)
ax.set_ylabel("ESH (hours)"); ax.set_title("Cumulative ESH by Face (all masks applied)")
ax.legend(ncol=3); _fmt_ax(ax); plt.tight_layout()
plt.savefig("outputs/fig_cumulative_esh_faces.png", dpi=150, bbox_inches="tight")
plt.show()


#### Cumulative ESH — All Faces

Full simulation cumulative ESH for all six faces. Slope changes mark eclipse entry/exit and structural blocking events.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name in face_normals.keys():
    ax.plot(df["t"], df[f"ESH_{name}_cum"], label=name)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Cumulative ESH (hours)")
ax.set_title("Cumulative Equivalent Sun Hours per MSC Face")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
plt.tight_layout()
plt.savefig("outputs/fig_cumulative_esh_all.png", dpi=150, bbox_inches="tight")
plt.show()


#### Per-Orbit Solar Flux

Masked total irradiance for all faces over one full ISS orbit (~100 min).
Shows the combined effect of geometry, eclipse, and structural masking.

In [ ]:
orbit_minutes = 100
orbit_steps   = int(orbit_minutes * 60 / df["dt"].iloc[0])
orbit_df      = df.iloc[:orbit_steps]

fig, ax = plt.subplots(figsize=(10, 5))
for name in face_normals.keys():
    ax.plot(
        (orbit_df.index - orbit_df.index[0]) * df["dt"].iloc[0] / 60,
        orbit_df[f"I_{name}"],
        label=name,
    )
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Irradiance (W/m²)")
ax.set_title("Simulated Solar Flux per MSC Face (1 Orbit)")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/fig_per_orbit_flux.png", dpi=150, bbox_inches="tight")
plt.show()


### Export for Downstream Pipeline

Write `prototype_face_irradiance_1day.csv`, consumed by `sensor_channel_split.ipynb`.

**Required columns (do not rename or remove):**

| Column | Description |
|---|---|
| `t` | Timestamp |
| `I_global` | Total masked irradiance for the export face (W/m²) |
| `export_face` | Which face is exported (`PRIMARY_FACE`) |
| `sun_factor` | Eclipse factor alias — do not rename |
| `I_global_sunmasked` | Backward-compatible alias for `I_global`. Now reflects **all three masks** (eclipse × structural × exposure). Column name retained to avoid breaking downstream. |

Additional diagnostic columns are appended but ignored by the downstream notebook.

The helper `build_face_export_df(df, face)` (defined in the cell below) assembles the dataframe without writing it — useful for future backend/HTML integration.

#### Export Helper: `build_face_export_df`

Assembles the downstream-compatible export dataframe for a given face without writing the CSV. Separating assembly from the write call makes it easy to reuse from a future backend or to inspect the dataframe before export.

In [ ]:
def build_face_export_df(df, face, valid_faces=None):
    """
    Assemble the downstream-compatible export dataframe for a given MSC face.

    Parameters
    ----------
    df : pd.DataFrame
        Master dataframe with all mask columns and irradiance columns computed.
    face : str
        Face label, e.g. "+H".
    valid_faces : list of str, optional
        If provided, validates that `face` is in the list.

    Returns
    -------
    pd.DataFrame with columns:
        t, I_global, export_face, sun_factor, I_global_sunmasked
        + diagnostic columns if present in df.

    Notes
    -----
    Column names are frozen for downstream compatibility (sensor_channel_split.ipynb).
    I_global_sunmasked reflects the full mask stack:
        eclipse_factor x structural_factor x exposure_open_factor.
    """
    if valid_faces is not None and face not in valid_faces:
        raise ValueError(
            f"face '{face}' is not valid. Choose one of: {valid_faces}"
        )

    face_col   = f"I_{face}"
    direct_col = f"I_direct_{face}"
    albedo_col = f"I_albedo_{face}"

    if face_col not in df.columns:
        raise KeyError(
            f"'{face_col}' not found in df. "
            f"Run the irradiance + masking cells first."
        )

    # Core columns — required by sensor_channel_split.ipynb
    out = pd.DataFrame({
        "t":           df["t"],
        "I_global":    df[face_col],        # total after all masks
        "export_face": face,
        "sun_factor":  df["sun_factor"],    # eclipse_factor alias — DO NOT RENAME
    })

    # I_global_sunmasked: backward-compatible alias for I_global.
    # Reflects all active masks: eclipse x structural x exposure_open_factor.
    # Name is kept unchanged to avoid breaking the downstream notebook.
    if direct_col in df.columns and albedo_col in df.columns:
        out["I_global_sunmasked"] = (
            df[direct_col] * df["eclipse_factor"] * df["structural_factor"] * df["exposure_open_factor"]
            + df[albedo_col] * df["eclipse_factor"] * df["exposure_open_factor"]
        )
    else:
        # Fallback: use the already-masked total column
        out["I_global_sunmasked"] = df[face_col]

    # Diagnostic columns — appended for analysis; ignored by downstream notebook
    for col in ["lighting_state", "eclipse_factor",
                "sun_az_deg", "sun_el_deg",
                "structural_blocked", "structural_factor",
                "exposure_open_factor"]:
        if col in df.columns:
            out[col] = df[col]

    return out


In [ ]:
# --- EXPORT: Prototype "global intensity vs time" for sensor-channel notebook ---
# Calls build_face_export_df() to assemble the dataframe, then writes the CSV.
# The function is defined in the cell above — modify schema there, not here.
#
# Required downstream columns (sensor_channel_split.ipynb reads these):
#   t, I_global, export_face, sun_factor, I_global_sunmasked
#
# Additional diagnostic columns are appended but do NOT affect downstream.

EXPORT_PATH = "prototype_face_irradiance_1day.csv"
EXPORT_FACE = PRIMARY_FACE   # set in User Configuration section above

export_df = build_face_export_df(df, EXPORT_FACE, valid_faces=VALID_FACES)
export_df.to_csv(EXPORT_PATH, index=False)

print(f"Exported {len(export_df)} rows → {EXPORT_PATH}")
print("Export face:", EXPORT_FACE)
print("Columns:", list(export_df.columns))
export_df.head()
